# JobMatch AI — Modelo Classificador Fit/No Fit

Este notebook:
1. Carrega dados processados
2. Pré-processa e vetoriza pares currículo-vaga
3. Avalia 3 classificadores via cross-validation (F1)
4. Treina o melhor modelo
5. Avalia no conjunto de teste

---

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

from src.pipeline.preprocess import clean_text
from src.models.vectorizer import fit_vectorizer, transform, load_vectorizer
from src.models.classifier import train_best, predict

print(f"📂 Projeto: {project_root}")

---
## 1. Carregar Dados Processados

In [ ]:
PROC = project_root / 'data' / 'processed'
pairs_path = PROC / 'training_pairs.parquet'

if not pairs_path.exists():
    raise FileNotFoundError(f"Arquivo não encontrado: {pairs_path}")

pairs = pd.read_parquet(pairs_path)
print(f"✅ Pares carregados: {pairs.shape}")
print(f"   Labels: {pairs['label'].value_counts().to_dict()}")

---
## 2. Pré-processamento e Vetorização

In [ ]:
pairs['resume_clean'] = pairs['resume'].fillna('').apply(clean_text)
pairs['job_clean'] = pairs['job_description'].fillna('').apply(clean_text)
pairs['combined'] = pairs['resume_clean'] + ' ' + pairs['job_clean']

corpus = pairs['combined'].tolist()
y = (pairs['label'] == 'Fit').astype(int).values

print(f"📊 Corpus: {len(corpus)} documentos")
print(f"📊 Target: {y.sum()} Fit / {(1-y).sum()} No Fit")

In [ ]:
vec = fit_vectorizer(corpus)
X = transform(corpus, vec)
print(f"✅ Matriz TF-IDF: {X.shape}")

---
## 3. Treinamento e Avaliação

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42,
)
print(f"Treino: {X_train.shape[0]} amostras")
print(f"Teste:  {X_test.shape[0]} amostras")

In [ ]:
best_name, best_model = train_best(X_train, y_train)

In [ ]:
y_pred = best_model.predict(X_test)
prob_pred = best_model.predict_proba(X_test)[:, 1] if hasattr(best_model, 'predict_proba') else y_pred

print(f"📊 Relatório de Classificação ({best_name}):\n")
print(classification_report(y_test, y_pred, target_names=['No Fit', 'Fit']))

# Matriz de Confusão
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['No Fit', 'Fit'], yticklabels=['No Fit', 'Fit'])
plt.title(f'Matriz de Confusão - {best_name}')
plt.ylabel('Real')
plt.xlabel('Predito')
plt.show()

In [ ]:
# Teste com exemplo
resume_example = "Data Scientist with Python, SQL, machine learning, and 5 years experience"
resume_clean = clean_text(resume_example)
resume_vec = transform([resume_clean], vec)
label, prob = predict(resume_vec, best_model)
print(f"🎯 Exemplo: {label} (prob={prob:.2%})")